In [5]:
import glob

folder = "/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/"
files = glob.glob(folder + "*.csv")
print(f"Files found: {len(files)}")
for f in sorted(files):
    print(f)

Files found: 30
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00000.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00001.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00002.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00003.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00004.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00005.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00006.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00007.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/S2TS_Maha_OC_00008.csv
/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseri

In [6]:
import pandas as pd
import glob
import os

folder = "/Users/aditibharadwaj/Documents/experiqs/MH_SOC/SHC_Maharashtra_S2_timeseries/"

files = glob.glob(folder + "*.csv")
print(f"Files found: {len(files)}")

df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in files],
    ignore_index=True
)

# ── BASIC CLEANING ───────────────────────────────────────────
band_cols = ["B2", "B3", "B4", "B8", "B11", "B12"]
before = len(df)
df = df.dropna(subset=band_cols, how="all")
print(f"Dropped {before - len(df)} rows with no band values")

# Convert dates
df["tile_date"] = pd.to_datetime(df["tile_date"])
df["sample_date_clean"] = pd.to_datetime(df["sample_date_clean"])

# Sort by point then chronologically — critical for LSTM
df = df.sort_values(["id", "tile_date"]).reset_index(drop=True)

# ── SANITY CHECK ─────────────────────────────────────────────
print(f"\nTotal rows: {len(df):,}")
print(f"Unique soil sample points: {df['id'].nunique():,}")
print(f"Avg observations per point: {len(df) / df['id'].nunique():.1f}")
print(f"Date range: {df['tile_date'].min()} to {df['tile_date'].max()}")
print(f"OC range: {df['OC'].min():.3f} to {df['OC'].max():.3f}")
print(f"\nNull counts:\n{df[band_cols + ['SCL', 'OC']].isnull().sum()}")

# ── SAVE ─────────────────────────────────────────────────────
output = "/Users/aditibharadwaj/Documents/experiqs/MH_SOC/S2_timeseries_combined.csv"
df.to_csv(output, index=False)
print(f"\nSaved: {output}")
print(f"File size: {os.path.getsize(output)/1e6:.1f} MB")

Files found: 30
Dropped 14740 rows with no band values

Total rows: 4,792,639
Unique soil sample points: 29,942
Avg observations per point: 160.1
Date range: 2021-10-23 00:00:00 to 2025-09-07 00:00:00
OC range: 0.007 to 3.500

Null counts:
B2     3340
B3     1359
B4      161
B8        9
B11       2
B12       7
SCL       9
OC        0
dtype: int64

Saved: /Users/aditibharadwaj/Documents/experiqs/MH_SOC/S2_timeseries_combined.csv
File size: 666.4 MB
